In [1]:
import json
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import T5Tokenizer, T5ForConditionalGeneration, get_linear_schedule_with_warmup
from torch.optim import AdamW

In [2]:
from google.colab import drive, runtime
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
%cd /content/drive/MyDrive/rocky-chatbot/
!ls

/content/drive/MyDrive/rocky-chatbot
train.json  val.json


In [5]:
MODEL_NAME     = "t5-base"
TRAIN_PATH     = "train.json"
VAL_PATH       = "val.json"
OUTPUT_DIR     = "rocky-t5"
MAX_INPUT_LEN  = 256
MAX_TARGET_LEN = 128
BATCH_SIZE     = 8      # reduce to 4 if you get OOM errors
EPOCHS         = 5
LEARNING_RATE  = 3e-4
WARMUP_STEPS   = 50
SAVE_EVERY     = 1      # save checkpoint every N epochs

In [6]:
class RockyDataset(Dataset):
    def __init__(self, path, tokenizer, max_input_len, max_target_len):
        with open(path, encoding="utf-8") as f:
            self.pairs = json.load(f)
        self.tokenizer     = tokenizer
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair = self.pairs[idx]

        inputs = self.tokenizer(
            pair["input_text"],
            max_length=self.max_input_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        targets = self.tokenizer(
            pair["target_text"],
            max_length=self.max_target_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        # Replace padding token id with -100 so it's ignored in loss
        labels = targets["input_ids"].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids":      inputs["input_ids"].squeeze(),
            "attention_mask": inputs["attention_mask"].squeeze(),
            "labels":         labels,
        }

In [7]:
def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    print(f"Loading {MODEL_NAME}...")
    tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
    model     = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
    model.to(device)

    print("Loading datasets...")
    train_dataset = RockyDataset(TRAIN_PATH, tokenizer, MAX_INPUT_LEN, MAX_TARGET_LEN)
    val_dataset   = RockyDataset(VAL_PATH,   tokenizer, MAX_INPUT_LEN, MAX_TARGET_LEN)
    print(f"  {len(train_dataset)} train / {len(val_dataset)} val")

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=WARMUP_STEPS,
        num_training_steps=total_steps,
    )

    best_val_loss = float("inf")

    for epoch in range(1, EPOCHS + 1):
        # --- Train ---
        model.train()
        train_loss = 0.0
        for step, batch in enumerate(train_loader):
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )
            loss = outputs.loss
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            train_loss += loss.item()

            if (step + 1) % 10 == 0:
                avg = train_loss / (step + 1)
                print(f"  Epoch {epoch} step {step+1}/{len(train_loader)} loss={avg:.4f}")

        avg_train_loss = train_loss / len(train_loader)

        # --- Validate ---
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                input_ids      = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels         = batch["labels"].to(device)
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels,
                )
                val_loss += outputs.loss.item()

        avg_val_loss = val_loss / len(val_loader)
        print(f"\nEpoch {epoch} — train loss: {avg_train_loss:.4f} | val loss: {avg_val_loss:.4f}\n")

        # --- Save ---
        if epoch % SAVE_EVERY == 0:
            checkpoint_dir = f"{OUTPUT_DIR}/checkpoint-epoch-{epoch}"
            model.save_pretrained(checkpoint_dir)
            tokenizer.save_pretrained(checkpoint_dir)
            print(f"Saved checkpoint to {checkpoint_dir}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            model.save_pretrained(f"{OUTPUT_DIR}/best")
            tokenizer.save_pretrained(f"{OUTPUT_DIR}/best")
            print(f"New best val loss {best_val_loss:.4f} — saved to {OUTPUT_DIR}/best")

    print("\nTraining complete.")
    print(f"Best val loss: {best_val_loss:.4f}")
    print(f"Best model saved to {OUTPUT_DIR}/best")

In [8]:
train()

Using device: cuda
Loading t5-base...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Saved checkpoint to rocky-t5/checkpoint-epoch-3
  Epoch 4 step 10/92 loss=2.1168
  Epoch 4 step 20/92 loss=2.2246
  Epoch 4 step 30/92 loss=2.2446
  Epoch 4 step 40/92 loss=2.2349
  Epoch 4 step 50/92 loss=2.2465
  Epoch 4 step 60/92 loss=2.2555
  Epoch 4 step 70/92 loss=2.2603
  Epoch 4 step 80/92 loss=2.2567
  Epoch 4 step 90/92 loss=2.2492

Epoch 4 — train loss: 2.2476 | val loss: 3.7726



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved checkpoint to rocky-t5/checkpoint-epoch-4
  Epoch 5 step 10/92 loss=1.9928
  Epoch 5 step 20/92 loss=2.0174
  Epoch 5 step 30/92 loss=1.9569
  Epoch 5 step 40/92 loss=1.9531
  Epoch 5 step 50/92 loss=1.9779
  Epoch 5 step 60/92 loss=1.9737
  Epoch 5 step 70/92 loss=1.9794
  Epoch 5 step 80/92 loss=1.9969
  Epoch 5 step 90/92 loss=1.9841

Epoch 5 — train loss: 1.9821 | val loss: 3.8707



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved checkpoint to rocky-t5/checkpoint-epoch-5

Training complete.
Best val loss: 3.5875
Best model saved to rocky-t5/best


In [9]:
runtime.unassign()